
# MCfd 四种近似的系统数值实验

这个 notebook 基于 `MCfd.ipynb` 中的 MC-I、MC-II、GJ-I、GJ-II 公式，把单元格脚本整理成可复用函数，并加入三组实验：

1. 光滑函数 $f(t)=e^{-t}$ 下，阶数 $\alpha$ 对数值分数阶导数和误差的影响；
2. 样本数/节点数 $M$ 对误差的影响，并加入 GJ 节点靠近端点导致抵消误差放大的诊断；
3. 非光滑函数 $f(t)=(t-0.7)_+^{1.25}$ 下的误差表现，用于检验理论中光滑性假设失效时的现象。

运行前请保证本 notebook 和 `mcfd_nature_analysis.py` 在同一目录下。


In [ ]:

%matplotlib inline
from pathlib import Path
import numpy as np

from mcfd_nature_analysis import (
    ExpFunction,
    HingePowerFunction,
    alpha_sweep,
    m_sweep,
    plot_alpha_sweep,
    plot_m_sweep,
    save_npz,
)

outdir = Path("mcfd_figures")
outdir.mkdir(exist_ok=True)



## 1. 光滑 benchmark：$f(t)=e^{-t}$，扫描 $\alpha$


In [ ]:

smooth = ExpFunction(lam=-1.0)

smooth_alpha = alpha_sweep(
    smooth,
    t=1.5,
    alphas=np.linspace(1.01, 1.99, 100),
    M_mc=10_000,
    M_gj=100,
    mc_repeats=8,
    seed=202601,
    stable_gj=False,
)
save_npz(smooth_alpha, outdir / "smooth_alpha_sweep.npz")
fig = plot_alpha_sweep(
    smooth_alpha,
    title="smooth benchmark: $f(t)=e^{-t}$",
    savepath=outdir / "smooth_alpha_sweep",
)
fig



## 2. 光滑 benchmark：扫描 $M$ 并诊断 GJ 误差放大

右图中的 $\tau_{\min}$ 是 GJ 节点映射到 $[0,1]$ 后的最小节点。随着 $M$ 增大，$\tau_{\min}$ 快速靠近 0。原始公式中会计算

\[
\frac{f'(t)-f'(t-h)}{h},\qquad
\frac{f(t)-f(t-h)-h f'(t)}{h^2}, \quad h=t\tau,
\]

所以当 $h$ 很小时，浮点抵消误差会分别被 $1/h$ 和 $1/h^2$ 放大。图中也给了 stable 版本：只对很小的 $h$ 用 Taylor 展开替代直接相减，用来区分“数学 quadrature 误差”和“浮点抵消误差”。


In [ ]:

smooth_M = m_sweep(
    smooth,
    t=1.5,
    alpha=1.5,
    Ms=np.array([10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120, 10240]),
    mc_repeats=32,
    max_gj_M=640,
    seed=202602,
    include_stable_gj=True,
)
save_npz(smooth_M, outdir / "smooth_M_sweep.npz")
fig = plot_m_sweep(
    smooth_M,
    alpha=1.5,
    title="smooth benchmark",
    savepath=outdir / "smooth_M_sweep",
    include_stable_gj=True,
)
fig



## 3. 非光滑 benchmark：$f(t)=(t-0.7)_+^{1.25}$

该函数在 $t=0.7$ 处只有 $C^1$ 正则性，不满足常规二阶光滑假设。它的 Caputo 导数有闭式表达：

\[
D_C^\alpha (t-c)_+^\beta
= \frac{\Gamma(\beta+1)}{\Gamma(\beta+1-\alpha)}(t-c)_+^{\beta-\alpha},\qquad \beta>1.
\]


In [ ]:

nonsmooth = HingePowerFunction(beta=1.25, c=0.7)

nonsmooth_alpha = alpha_sweep(
    nonsmooth,
    t=1.5,
    alphas=np.linspace(1.01, 1.95, 90),
    M_mc=20_000,
    M_gj=120,
    mc_repeats=8,
    seed=202603,
    stable_gj=False,
)
save_npz(nonsmooth_alpha, outdir / "nonsmooth_alpha_sweep.npz")
fig = plot_alpha_sweep(
    nonsmooth_alpha,
    title=r"non-smooth benchmark: $f(t)=(t-0.7)_+^{1.25}$",
    savepath=outdir / "nonsmooth_alpha_sweep",
)
fig


In [ ]:

nonsmooth_M = m_sweep(
    nonsmooth,
    t=1.5,
    alpha=1.5,
    Ms=np.array([10, 20, 40, 80, 160, 320, 640, 1280, 2560]),
    mc_repeats=32,
    max_gj_M=640,
    seed=202604,
    include_stable_gj=False,
)
save_npz(nonsmooth_M, outdir / "nonsmooth_M_sweep.npz")
fig = plot_m_sweep(
    nonsmooth_M,
    alpha=1.5,
    title=r"non-smooth benchmark",
    savepath=outdir / "nonsmooth_M_sweep",
    include_stable_gj=False,
)
fig



## 可调参数建议

- 若需要复现更接近原图的 MC 抖动，把 `mc_repeats` 改成 1。
- 若需要更稳定的 MC 曲线，增大 `M_mc` 或 `mc_repeats`。
- 若希望专门观察 GJ 随 $M$ 增大后的反常放大，把 `max_gj_M` 增大；如果 `roots_jacobi` 对大 $M$ 变慢或出现 NaN，这是数值病态本身的一部分。
